In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=8, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [4]:
from src.configs import (S, C, B, IDX_TO_CLASS, CONFIDENCE_THRESHOLD,
                         NMS_IOU_THRESHOLD)
from src.utils import convert_xywh_coords, IoU
from operator import itemgetter

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]

                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False)

                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

def filter_group_sort_preds(decoded_preds):
    sorted_preds = []

    # 1. filter and group remaining predictions by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]

        sorted_preds.append(valid_preds)

    # 2. sort each class's predictions by confidence score
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1))

    return sorted_preds


def NMS(preds_batch):
    # 1. decode batch of predictions
    decoded_preds = decode_preds(preds_batch)

    # 2. filter, group, and sort the decoded predictions
    sorted_preds = filter_group_sort_preds(decoded_preds)

    # 3. perform Non-Maximum Suppression
    final_preds = []

    for image in sorted_preds:
        final_img_preds = {}

        for class_name, preds in image.items():
            final_img_preds[class_name] = []

            while preds:
                highest_conf = preds.pop(0)
                final_img_preds[class_name].append(highest_conf)

                preds = [pred for pred in preds if
                         IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

        final_preds.append(final_img_preds)

    return final_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

preds = model(X_batch)
preds.shape

torch.Size([8, 1470])

In [6]:
decoded_preds = decode_preds(preds)
decoded_preds

[[('sheep',
   -0.1483011522565505,
   tensor(-1.1144, grad_fn=<SubBackward0>),
   tensor(-12.5478, grad_fn=<SubBackward0>),
   tensor(27.4272, grad_fn=<AddBackward0>),
   tensor(16.6444, grad_fn=<AddBackward0>)),
  ('sheep',
   -0.09083753170390807,
   tensor(0.5554, grad_fn=<SubBackward0>),
   tensor(-20.9205, grad_fn=<SubBackward0>),
   tensor(-12.9006, grad_fn=<AddBackward0>),
   tensor(47.9006, grad_fn=<AddBackward0>)),
  ('sofa',
   -0.08330767742928025,
   tensor(5.4791, grad_fn=<SubBackward0>),
   tensor(-24.7853, grad_fn=<SubBackward0>),
   tensor(29.2145, grad_fn=<AddBackward0>),
   tensor(11.2935, grad_fn=<AddBackward0>)),
  ('sofa',
   -0.09384901487125141,
   tensor(39.9227, grad_fn=<SubBackward0>),
   tensor(28.4590, grad_fn=<SubBackward0>),
   tensor(29.0169, grad_fn=<AddBackward0>),
   tensor(-22.6720, grad_fn=<AddBackward0>)),
  ('diningtable',
   -0.19769966017586071,
   tensor(62.1465, grad_fn=<SubBackward0>),
   tensor(6.2842, grad_fn=<SubBackward0>),
   tensor(62.3

In [7]:
sorted_preds = filter_group_sort_preds(decoded_preds)
sorted_preds

[{'bicycle': [('bicycle',
    0.3972091889864373,
    tensor(-1.1401, grad_fn=<SubBackward0>),
    tensor(130.3733, grad_fn=<SubBackward0>),
    tensor(6.6261, grad_fn=<AddBackward0>),
    tensor(83.7014, grad_fn=<AddBackward0>))]},
 {},
 {'horse': [('horse',
    0.45379896785476603,
    tensor(187.3075, grad_fn=<SubBackward0>),
    tensor(164.3748, grad_fn=<SubBackward0>),
    tensor(52.0257, grad_fn=<AddBackward0>),
    tensor(-5.7140, grad_fn=<AddBackward0>)),
   ('horse',
    0.617067027363305,
    tensor(137.0080, grad_fn=<SubBackward0>),
    tensor(1.3637, grad_fn=<SubBackward0>),
    tensor(166.1828, grad_fn=<AddBackward0>),
    tensor(18.7620, grad_fn=<AddBackward0>))],
  'motorbike': [('motorbike',
    0.623768143129233,
    tensor(195.2209, grad_fn=<SubBackward0>),
    tensor(141.3487, grad_fn=<SubBackward0>),
    tensor(124.5827, grad_fn=<AddBackward0>),
    tensor(105.8754, grad_fn=<AddBackward0>))],
  'car': [('car',
    0.3798910246990683,
    tensor(14.7194, grad_fn=<Sub

In [8]:
import torch 
nums = torch.arange(10)
nums = nums.sort()[0]

print(nums)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [9]:
nums = [num for num in nums if num == 2 or num == 4 or num == 6]
nums

[tensor(2), tensor(4), tensor(6)]

In [10]:
final_preds = []

for image in sorted_preds:
    final_img_preds = {}

    for class_name, preds in image.items():
        final_img_preds[class_name] = []

        while preds:
            highest_conf = preds.pop(0)
            final_img_preds[class_name].append(highest_conf)

            preds = [pred for pred in preds if
                     IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

    final_preds.append(final_img_preds)

In [11]:
final_preds, len(final_preds)

([{'bicycle': [('bicycle',
     0.3972091889864373,
     tensor(-1.1401, grad_fn=<SubBackward0>),
     tensor(130.3733, grad_fn=<SubBackward0>),
     tensor(6.6261, grad_fn=<AddBackward0>),
     tensor(83.7014, grad_fn=<AddBackward0>))]},
  {},
  {'horse': [('horse',
     0.45379896785476603,
     tensor(187.3075, grad_fn=<SubBackward0>),
     tensor(164.3748, grad_fn=<SubBackward0>),
     tensor(52.0257, grad_fn=<AddBackward0>),
     tensor(-5.7140, grad_fn=<AddBackward0>)),
    ('horse',
     0.617067027363305,
     tensor(137.0080, grad_fn=<SubBackward0>),
     tensor(1.3637, grad_fn=<SubBackward0>),
     tensor(166.1828, grad_fn=<AddBackward0>),
     tensor(18.7620, grad_fn=<AddBackward0>))],
   'motorbike': [('motorbike',
     0.623768143129233,
     tensor(195.2209, grad_fn=<SubBackward0>),
     tensor(141.3487, grad_fn=<SubBackward0>),
     tensor(124.5827, grad_fn=<AddBackward0>),
     tensor(105.8754, grad_fn=<AddBackward0>))],
   'car': [('car',
     0.3798910246990683,
     

In [12]:
y_batch.shape, y_batch, y_batch[0].shape

(torch.Size([8, 7, 7, 30]),
 tensor([[[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.000

In [13]:
(y_batch[0].flatten(0, 1))[29]

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.7500, 0.5938, 0.2232, 0.4821, 1.0000, 0.7500, 0.5938,
        0.2232, 0.4821, 1.0000])

In [14]:
from src.configs import IDX_TO_CLASS

def find_objects(y_batch, device):
    y_batch = y_batch.flatten(1, 2)
    truth_objects = []

    for target in y_batch:
        class_objects = {}
        
        for cell in target:
            class_name = IDX_TO_CLASS[int(torch.argmax(cell[:C]))]

            if cell[C+4] == 1:
                # the additional 0 is to classify that specific object as 
                # unmatched with a prediction. 1 is for matched. 
                bboxes = torch.cat((cell[C:C+4], torch.tensor([0]).to(device)))
                if class_name in class_objects:
                    class_objects[class_name].append(bboxes)
                else:
                    class_objects[class_name] = [bboxes]

        truth_objects.append(class_objects)
        
    return truth_objects

truth_objects = find_objects(y_batch, "cpu")

In [15]:
len(truth_objects)

8

In [16]:
final_preds[0]

{'bicycle': [('bicycle',
   0.3972091889864373,
   tensor(-1.1401, grad_fn=<SubBackward0>),
   tensor(130.3733, grad_fn=<SubBackward0>),
   tensor(6.6261, grad_fn=<AddBackward0>),
   tensor(83.7014, grad_fn=<AddBackward0>))]}

In [17]:
truth_objects

[{'person': [tensor([0.7188, 0.9375, 0.1250, 0.4375, 0.0000]),
   tensor([0.5625, 0.4688, 0.1071, 0.4509, 0.0000])],
  'horse': [tensor([0.7500, 0.5938, 0.2232, 0.4821, 0.0000]),
   tensor([0.0938, 0.0000, 0.4196, 0.5670, 0.0000])]},
 {'chair': [tensor([0.1562, 0.1875, 0.1696, 0.2455, 0.0000])],
  'sofa': [tensor([0.3125, 0.6250, 0.4732, 0.3929, 0.0000])]},
 {'motorbike': [tensor([0.0625, 0.0625, 0.5179, 0.8080, 0.0000]),
   tensor([0.5312, 0.6250, 0.4241, 0.3616, 0.0000])]},
 {'chair': [tensor([0.4375, 0.3750, 0.1027, 0.0893, 0.0000]),
   tensor([0.9375, 0.3750, 0.0402, 0.0357, 0.0000]),
   tensor([0.5000, 0.6250, 0.0402, 0.0938, 0.0000]),
   tensor([0.6562, 0.7188, 0.0759, 0.0982, 0.0000]),
   tensor([0.4375, 0.8125, 0.1027, 0.1161, 0.0000]),
   tensor([0.1875, 0.6250, 0.0714, 0.3080, 0.0000]),
   tensor([0.5000, 0.8750, 0.1250, 0.3080, 0.0000]),
   tensor([0.4375, 0.5625, 0.1652, 0.1295, 0.0000])],
  'diningtable': [tensor([0.8125, 0.9062, 0.6339, 0.3125, 0.0000])]},
 {'aeroplane': 

In [18]:
truth_objects[0]

{'person': [tensor([0.7188, 0.9375, 0.1250, 0.4375, 0.0000]),
  tensor([0.5625, 0.4688, 0.1071, 0.4509, 0.0000])],
 'horse': [tensor([0.7500, 0.5938, 0.2232, 0.4821, 0.0000]),
  tensor([0.0938, 0.0000, 0.4196, 0.5670, 0.0000])]}

In [19]:
TP_IOU_THRESHOLD = 0.5

def tp_fp_and_count_objects(final_preds, truth_objects, all_tp_fp_by_class, class_object_totals):
    # 1. iterate through each image prediction/label in the batch
    for b in range(len(final_preds)):
        img_preds = final_preds[b]
        objects = truth_objects[b]

        # 2. iterate through each class
        for class_name, preds in img_preds.items():            
            # a. if any of the ground truth objects belong to the class
            if class_name in objects:
                # iterate through each prediction, find max IoU truth object, and
                # if the max IoU surpasses the threshold, it is a TP, else FP
                for pred in preds:
                    IoUs = [IoU(object_[0:4], pred[2:6]) for object_ in objects[class_name]]
                    max_idx = IoUs.index(max(IoUs))
                    
                    if IoUs[max_idx] < TP_IOU_THRESHOLD:
                        all_tp_fp_by_class[class_name].append((pred[1], False))
                        
                    elif objects[class_name][max_idx][-1] == 0:
                        all_tp_fp_by_class[class_name].append((pred[1], True))
                        objects[class_name][max_idx][-1] = 1
                        
                    else:
                        all_tp_fp_by_class[class_name].append((pred[1], False))
            else:
                # b. all predictions belonging to class are FP since there are
                # no ground truth objects belonging to that class
                for pred in preds:
                    all_tp_fp_by_class[class_name].append((pred[1], False))    

        # 3. add the number of objects that belong to each class to the total
        for class_name, object_list in objects.items():
            class_object_totals[class_name] += len(object_list)

In [20]:
all_tp_fp_by_class = {
    "aeroplane": [],
    "bicycle": [],
    "bird": [],
    "boat": [],
    "bottle": [],
    "bus": [],
    "car": [],
    "cat": [],
    "chair": [],
    "cow":[],
    "diningtable": [],
    "dog": [],
    "horse": [],
    "motorbike": [],
    "person": [],
    "pottedplant": [],
    "sheep": [],
    "sofa": [],
    "train": [],
    "tvmonitor": [],
}

precision_recall_lists = {
    "aeroplane": ([], []),
    "bicycle": ([], []),
    "bird": ([], []),
    "boat": ([], []),
    "bottle": ([], []),
    "bus": ([], []),
    "car": ([], []),
    "cat": ([], []),
    "chair": ([], []),
    "cow":([], []),
    "diningtable": ([], []),
    "dog": ([], []),
    "horse": ([], []),
    "motorbike": ([], []),
    "person": ([], []),
    "pottedplant": ([], []),
    "sheep": ([], []),
    "sofa": ([], []),
    "train": ([], []),
    "tvmonitor": ([], []),
}

class_object_totals = {
    "aeroplane": 0,
    "bicycle": 0,
    "bird": 0,
    "boat": 0,
    "bottle": 0,
    "bus": 0,
    "car": 0,
    "cat": 0,
    "chair": 0,
    "cow":0,
    "diningtable": 0,
    "dog": 0,
    "horse": 0,
    "motorbike": 0,
    "person": 0,
    "pottedplant": 0,
    "sheep": 0,
    "sofa": 0,
    "train": 0,
    "tvmonitor": 0,
}

In [21]:
tp_fp_and_count_objects(final_preds, truth_objects, all_tp_fp_by_class, class_object_totals)

all_tp_fp_by_class["bird"].sort(key=itemgetter(0), reverse=True)
all_tp_fp_by_class, class_object_totals

({'aeroplane': [],
  'bicycle': [(0.3972091889864373, False)],
  'bird': [],
  'boat': [],
  'bottle': [(0.41608576059505076, False)],
  'bus': [],
  'car': [(0.3798910246990683, False), (0.4003854379453653, False)],
  'cat': [],
  'chair': [],
  'cow': [(0.7794804294900928, False)],
  'diningtable': [],
  'dog': [],
  'horse': [(0.45379896785476603, False), (0.617067027363305, False)],
  'motorbike': [(0.623768143129233, False)],
  'person': [(0.420407372712976, False)],
  'pottedplant': [],
  'sheep': [],
  'sofa': [],
  'train': [],
  'tvmonitor': []},
 {'aeroplane': 1,
  'bicycle': 0,
  'bird': 0,
  'boat': 2,
  'bottle': 0,
  'bus': 0,
  'car': 0,
  'cat': 0,
  'chair': 9,
  'cow': 0,
  'diningtable': 1,
  'dog': 0,
  'horse': 2,
  'motorbike': 2,
  'person': 5,
  'pottedplant': 0,
  'sheep': 0,
  'sofa': 1,
  'train': 0,
  'tvmonitor': 0})

In [23]:
def mean_average_precision(all_tp_fp_by_class, class_object_totals, precision_recall_lists):
    ap_by_class = {}
    
    for class_name, tp_fp_list in all_tp_fp_by_class.items():
        if class_object_totals[class_name] == 0:
            continue 
            
        AP = 0
        
        # 1. sort every list of TP/FPs in each class
        tp_fp_list.sort(key=itemgetter(0), reverse=True)

        # 2. iterate through list of TP/FPs and compute precision/recall
        true_positives = 0
        
        precision_denom = 0 # denominator - <total TP or FP>
        recall_denom = class_object_totals[class_name] # denominator - <total objects in class>
        
        previous_recall = 0

        precision_list = precision_recall_lists[class_name][0]
        recall_list = precision_recall_lists[class_name][1]

        for _, status in tp_fp_list:
            true_positives += status
            precision_denom += 1
            
            precision = true_positives / precision_denom
            recall = true_positives / recall_denom

            precision_list.append(precision)
            recall_list.append(recall)

            delta_recall = recall - previous_recall
            AP += precision * delta_recall

            previous_recall = recall

        ap_by_class[class_name] = AP

    mAP = sum(ap_by_class.values()) / len(ap_by_class)

    return mAP, ap_by_class

In [24]:
mAP, ap_by_class =  mean_average_precision(all_tp_fp_by_class, class_object_totals, precision_recall_lists)
mAP, ap_by_class

(0.0,
 {'aeroplane': 0,
  'boat': 0,
  'chair': 0,
  'diningtable': 0,
  'horse': 0.0,
  'motorbike': 0.0,
  'person': 0.0,
  'sofa': 0})

In [25]:
precision_recall_lists

{'aeroplane': ([], []),
 'bicycle': ([], []),
 'bird': ([], []),
 'boat': ([], []),
 'bottle': ([], []),
 'bus': ([], []),
 'car': ([], []),
 'cat': ([], []),
 'chair': ([], []),
 'cow': ([], []),
 'diningtable': ([], []),
 'dog': ([], []),
 'horse': ([0.0, 0.0], [0.0, 0.0]),
 'motorbike': ([0.0], [0.0]),
 'person': ([0.0], [0.0]),
 'pottedplant': ([], []),
 'sheep': ([], []),
 'sofa': ([], []),
 'train': ([], []),
 'tvmonitor': ([], [])}

In [26]:
len(precision_recall_lists)

20